In [ ]:
%%writefile requirements.txt
torch==2.1.2
torchvision==0.16.2
transformers==4.36.2
scikit-learn==1.3.2
pandas==2.1.4
numpy==1.26.2
matplotlib==3.8.2
seaborn==0.13.0
textblob==0.17.1
wordcloud==1.9.3
plotly==5.18.0
xgboost==2.0.3
nltk==3.8.1
Pillow==10.1.0
tqdm==4.66.1
python-dotenv==1.0.0
streamlit==1.29.0
fastapi==0.104.1
uvicorn==0.24.0
python-multipart==0.0.6
joblib==1.3.2

In [ ]:
# ============================================================
# CELL 1 — INSTALL LIBRARIES
# Sabse pehle yeh cell chalao
# ============================================================

get_ipython().system('pip install -q torch torchvision transformers scikit-learn')
get_ipython().system('pip install -q pandas numpy matplotlib seaborn textblob')
get_ipython().system('pip install -q wordcloud plotly xgboost nltk Pillow tqdm python-dotenv')
get_ipython().system('pip install -q streamlit')

import nltk
for r in ['stopwords', 'wordnet', 'punkt', 'averaged_perceptron_tagger', 'omw-1.4']:
    nltk.download(r, quiet=True)

print("=" * 50)
print("CELL 1 DONE — Libraries installed!")
print("=" * 50)


# ============================================================
# CELL 2 — CREATE FOLDER STRUCTURE
# Project ke liye zaroori folders banao
# ============================================================

import os

folders = [
    "data/raw/amazon_reviews",
    "data/raw/flipkart_reviews",
    "data/raw/waste_images",
    "data/processed",
    "models/nlp",
    "models/cnn",
    "models/rnn",
    "models/final",
    "reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("=" * 50)
print("CELL 2 DONE — Folder structure ready!")
print("=" * 50)
print()
for f in folders:
    print(f"  📁 {f}")

In [ ]:
# ============================================================
# CELL 3 — CREATE ALL DATASETS
# Flipkart Reviews + Amazon Reviews + Waste Images
# ============================================================

import numpy as np
import pandas as pd
import random
from pathlib import Path
from PIL import Image, ImageDraw


# ─────────────────────────────────────────────────────────────
# DATASET 1: FLIPKART PRODUCT REVIEWS
# ─────────────────────────────────────────────────────────────

def create_flipkart_dataset(output_path="data/raw/flipkart_reviews", n_samples=1000):

    products = [
        "EcoBottle Bamboo Water Bottle",
        "GreenLeaf Organic Green Tea",
        "NaturalGlow Organic Face Cream",
        "BambooToothbrush Pack of 4",
        "EarthKind Organic Cleaning Spray",
        "PlasticFree Beeswax Food Wrap",
        "OrganicCotton Reusable Bags",
        "SolarPowered Phone Charger",
        "CompostablePlates Pack 50",
        "RecycledPaper Notebook",
    ]

    genuine = [
        "Amazing product USDA certified organic and comes in completely plastic-free packaging. Love that they use renewable energy in manufacturing.",
        "This bamboo toothbrush is fantastic. B-Corp certified company truly sustainable. Packaging is 100 percent compostable.",
        "Finally a genuinely eco-friendly product! Third-party certified transparent supply chain info on website. Highly recommend.",
        "Excellent quality. Recycled materials clearly stated with certification numbers. Zero plastic in packaging.",
        "This organic product is legitimately certified by Green Seal. Been using 6 months great results and guilt-free purchase.",
        "Solar charger works perfectly. Company publishes annual sustainability report. Genuinely committed to environment.",
        "Biodegradable packaging verified by third party. Ingredients all organic and certified. Worth every rupee.",
        "Best eco product I have bought. Fair trade certified supports local farmers. Packaging is recycled cardboard only.",
        "Truly zero waste product. Everything from packaging to product itself is compostable. Company is carbon neutral certified.",
        "Love this brand. They show exact carbon footprint on packaging. Rainforest Alliance certified. Real sustainability!",
    ]

    greenwash = [
        "Says eco friendly but comes wrapped in layers of plastic! All natural claim has no certification anywhere on pack.",
        "Chemical free claim is ridiculous everything has chemicals. No eco certifications found despite green marketing.",
        "Advertised as sustainable but manufactured in coal powered factory. No proof of any environmental claims made.",
        "Claims to be organic but zero certification on packaging. Just green colored label does not make it organic.",
        "Natural ingredients claim is vague and unverified. Contains synthetic chemicals despite eco friendly branding.",
        "Biodegradable claim only works in industrial composting facility not home compost. Very misleading advertising.",
        "Green packaging is just regular plastic painted green. Self certified eco label means nothing without third party.",
        "Carbon neutral claim with no offset certificates published. Just marketing speak with no substance behind it.",
        "Sustainable sourcing claim but zero transparency on supply chain. Cannot verify any of their environmental claims.",
        "Pure and natural on label but ingredient list shows many synthetic additives. Classic greenwashing product.",
    ]

    np.random.seed(42)
    random.seed(42)
    records = []

    for i in range(n_samples):
        r = random.random()
        if r < 0.40:
            rev, rat, gw = random.choice(genuine), random.randint(4, 5), 0
        elif r < 0.75:
            rev, rat, gw = random.choice(greenwash), random.randint(1, 3), 1
        else:
            rev, rat, gw = random.choice(genuine[:3]), 3, 0

        yr  = random.randint(2021, 2024)
        mo  = random.randint(1, 12)
        dy  = random.randint(1, 28)

        records.append({
            'product_name'      : random.choice(products),
            'review'            : rev,
            'rating'            : rat,
            'timestamp'         : f"{yr}-{mo:02d}-{dy:02d}",
            'verified_purchase' : random.choice([True, True, False]),
            'helpful_votes'     : random.randint(0, 150),
            'is_greenwashing'   : gw,
            'category'          : random.choice(['Personal Care', 'Kitchen', 'Food', 'Home Care'])
        })

    df = pd.DataFrame(records)
    Path(output_path).mkdir(parents=True, exist_ok=True)
    df.to_csv(f"{output_path}/flipkart_reviews.csv", index=False)
    print(f"  ✅ Flipkart: {len(df)} reviews | Eco: {(df['is_greenwashing']==0).sum()} | Greenwash: {(df['is_greenwashing']==1).sum()}")
    return df


# ─────────────────────────────────────────────────────────────
# DATASET 2: AMAZON PRODUCT REVIEWS
# ─────────────────────────────────────────────────────────────

def create_amazon_dataset(output_path="data/raw/amazon_reviews", n_samples=1000):

    eco = [
        "This product is USDA certified organic. Packaging uses 100 percent recycled materials. Very impressed with transparency.",
        "B-Corp certified company. Solar powered facility. Truly eco friendly from production to packaging. Worth the price.",
        "Rainforest Alliance certified coffee. Fair trade verified. Carbon neutral shipping. Genuinely sustainable brand.",
        "Green Seal certified cleaning product. Plant based ingredients only. Refillable packaging option available. Love it.",
        "Certified compostable packaging. Organic ingredients verified by third party lab. Zero plastic anywhere. Excellent.",
        "FSC certified paper products. Wind powered manufacturing. Carbon offset program verified independently. Great brand.",
        "GOTS certified organic cotton. No synthetic dyes or chemicals. Transparent supply chain on their website. Recommend.",
        "Cradle to cradle certified product. Everything recyclable or compostable. Company publishes sustainability report.",
        "1 percent for Planet member. Uses 100 percent renewable energy. Plastic free packaging certified. Real commitment.",
        "EWG verified ingredients. No harmful chemicals. Leaping Bunny cruelty free certified. Genuinely safe product.",
    ]

    gw = [
        "All natural label but full of synthetic chemicals inside. No certifications found after research. Pure greenwashing.",
        "Green packaging is just marketing. Same plastic as before just colored green. No actual eco improvement made.",
        "Chemical free claim is scientifically impossible. Everything is chemicals. No certification to back any claims.",
        "Sustainable branding with zero transparency. Cannot find any sustainability report or certification anywhere online.",
        "Natural ingredients claim is vague. No definition of natural provided. No third party verification of claims.",
        "Eco friendly printed on plastic wrapped product. Ironic and misleading. Zero actual environmental consideration.",
        "Carbon neutral claim with no evidence. No offset certificates published. No third party verification anywhere.",
        "Organic label is self certified. No USDA or any other recognized certification. Just a marketing claim.",
        "Biodegradable only in industrial facility. Normal home compost cannot process this. Misleading advertising.",
        "Pure and natural on label. Ingredient list shows parabens and sulfates. Contradictory and dishonest marketing.",
    ]

    import datetime
    np.random.seed(123)
    random.seed(123)
    records = []

    for i in range(n_samples):
        r = random.random()
        if r < 0.45:
            rev, sc, gw_ = random.choice(eco), random.randint(4, 5), 0
        elif r < 0.80:
            rev, sc, gw_ = random.choice(gw), random.randint(1, 3), 1
        else:
            rev, sc, gw_ = random.choice(eco[:3]), 3, 0

        yr  = random.randint(2020, 2024)
        mo  = random.randint(1, 12)
        dy  = random.randint(1, 28)
        ts  = int(datetime.datetime(yr, mo, dy).timestamp())

        records.append({
            'Id'                     : i + 1,
            'ProductId'              : f"B{random.randint(10000000, 99999999):08d}",
            'UserId'                 : f"A{random.randint(100000, 999999)}",
            'ProfileName'            : f"User_{i}",
            'HelpfulnessNumerator'   : random.randint(0, 50),
            'HelpfulnessDenominator' : random.randint(1, 60),
            'Score'                  : sc,
            'Time'                   : ts,
            'Summary'                : rev[:60],
            'Text'                   : rev,
            'is_greenwashing'        : gw_
        })

    df = pd.DataFrame(records)
    Path(output_path).mkdir(parents=True, exist_ok=True)
    df.to_csv(f"{output_path}/Reviews.csv", index=False)
    print(f"  ✅ Amazon: {len(df)} reviews | Eco: {(df['is_greenwashing']==0).sum()} | Greenwash: {(df['is_greenwashing']==1).sum()}")
    return df


# ─────────────────────────────────────────────────────────────
# DATASET 3: WASTE CLASSIFICATION IMAGES
# ─────────────────────────────────────────────────────────────

def create_waste_images(output_path="data/raw/waste_images", n_train=200, n_test=60):

    for d in [
        f"{output_path}/DATASET/TRAIN/O",
        f"{output_path}/DATASET/TRAIN/R",
        f"{output_path}/DATASET/TRAIN/N", # Added 'N' for Non-Eco
        f"{output_path}/DATASET/TEST/O",
        f"{output_path}/DATASET/TEST/R",
        f"{output_path}/DATASET/TEST/N"  # Added 'N' for Non-Eco
    ]:
        Path(d).mkdir(parents=True, exist_ok=True)

    def make_organic():
        arr = np.zeros((224, 224, 3), dtype=np.uint8)
        b   = random.choice([(34, 139, 34), (101, 67, 33), (85, 107, 47)])
        for c in range(3):
            arr[:, :, c] = np.clip(b[c] + np.random.randint(-30, 30, (224, 224)), 0, 255)
        img  = Image.fromarray(arr)
        draw = ImageDraw.Draw(img)
        for _ in range(random.randint(3, 6)):
            x, y, r = random.randint(20, 200), random.randint(20, 200), random.randint(10, 35)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=(20, 160, 20), outline=(0, 80, 0))
        draw.rectangle([5, 5, 115, 28], fill=(255, 255, 255))
        draw.text((10, 8), "ORGANIC", fill=(0, 100, 0))
        return img

    def make_recyclable():
        arr = np.zeros((224, 224, 3), dtype=np.uint8)
        b   = random.choice([(33, 150, 243), (96, 125, 139), (144, 164, 174)])
        for c in range(3):
            arr[:, :, c] = np.clip(b[c] + np.random.randint(-25, 25, (224, 224)), 0, 255)
        img  = Image.fromarray(arr)
        draw = ImageDraw.Draw(img)
        for _ in range(random.randint(2, 5)):
            x1, y1 = random.randint(10, 150), random.randint(10, 150)
            draw.rectangle([x1, y1, x1+random.randint(20, 60), y1+random.randint(20, 60)],
                           fill=(0, 100, 200), outline=(0, 0, 150))
        draw.rectangle([5, 5, 130, 28], fill=(255, 255, 255))
        draw.text((10, 8), "RECYCLABLE", fill=(0, 0, 150))
        return img

    # Added a function to create non-eco images
    def make_non_eco():
        arr = np.zeros((224, 224, 3), dtype=np.uint8)
        b   = random.choice([(255, 0, 0), (100, 100, 100), (0, 0, 0)])
        for c in range(3):
            arr[:, :, c] = np.clip(b[c] + np.random.randint(-20, 20, (224, 224)), 0, 255)
        img  = Image.fromarray(arr)
        draw = ImageDraw.Draw(img)
        for _ in range(random.randint(4, 8)):
            x1, y1 = random.randint(10, 180), random.randint(10, 180)
            draw.line([x1, y1, x1+random.randint(20, 40), y1+random.randint(20, 40)],
                      fill=(255, 255, 255), width=random.randint(2, 5))
        draw.rectangle([5, 5, 115, 28], fill=(255, 255, 255))
        draw.text((10, 8), "NON-ECO", fill=(255, 0, 0))
        return img

    count = 0
    for split, n in [('TRAIN', n_train), ('TEST', n_test)]:
        # Generate for 'O', 'R', and 'N' (Non-Eco)
        for cls, fn in [('O', make_organic), ('R', make_recyclable), ('N', make_non_eco)]:
            for i in range(n // 2):
                fn().save(f"{output_path}/DATASET/{split}/{cls}/{cls}_{i:04d}.jpg", 'JPEG', quality=85)
                count += 1

    print(f"  ✅ Waste Images: {count} images created")
    print(f"     TRAIN/O: {n_train//2} | TRAIN/R: {n_train//2} | TRAIN/N: {n_train//2}") # Updated count
    print(f"     TEST/O:  {n_test//2}  | TEST/R:  {n_test//2}  | TEST/N: {n_test//2}") # Updated count


# ─────────────────────────────────────────────────────────────
# RUN — Sab datasets banao
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 3 — Creating All Datasets...")
print("=" * 50)

flipkart_df = create_flipkart_dataset(n_samples=1000)
amazon_df   = create_amazon_dataset(n_samples=1000)
create_waste_images(n_train=200, n_test=60)

print()
print("=" * 50)
print("CELL 3 DONE — All datasets ready!")
print("=" * 50)


In [ ]:
# ============================================================
# CELL 4 — DATA PREPROCESSING
# Text cleaning, feature engineering, dataset merging
# Is file ka code run karne se pehle cell3 zaroor chalao
# ============================================================

import re
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)


# ─────────────────────────────────────────────────────────────
# SUSTAINABILITY KEYWORD LEXICON
# Yeh words detect karte hain eco vs greenwash
# ─────────────────────────────────────────────────────────────

SUSTAINABILITY_LEXICON = {

    "strong_eco_keywords": {
        "certified organic": 3.0,
        "usda organic": 3.0,
        "fair trade certified": 3.0,
        "green seal certified": 3.0,
        "rainforest alliance": 3.0,
        "b corp": 3.0,
        "carbon neutral certified": 2.5,
        "zero waste certified": 2.5,
        "cradle to cradle": 2.5,
        "biodegradable": 2.0,
        "compostable": 2.0,
        "recycled materials": 2.0,
        "renewable energy": 2.0,
        "plastic free": 2.0,
        "zero plastic": 2.0,
    },

    "moderate_eco_keywords": {
        "eco-friendly": 1.5,
        "sustainable": 1.5,
        "environmentally friendly": 1.5,
        "organic": 1.2,
        "natural": 1.0,
        "green": 1.0,
        "recyclable": 1.5,
        "reusable": 1.5,
        "solar powered": 2.0,
        "carbon offset": 1.5,
        "low carbon": 1.5,
        "responsibly sourced": 1.5,
        "ethically sourced": 1.5,
    },

    "greenwashing_indicators": {
        "all natural": -1.5,
        "pure": -1.0,
        "non-toxic": -1.0,
        "chemical free": -2.0,
        "earth friendly": -1.0,
        "environmentally safe": -1.0,
        "going green": -0.5,
        "eco": -0.5,
        "bio": -0.5,
        "green packaging": -0.5,
        "sustainable choice": -0.5,
    },

    "contradiction_phrases": [
        r"organic.*plastic wrap",
        r"eco.friendly.*excessive packaging",
        r"green.*single.use",
        r"sustainable.*disposable",
        r"natural.*artificial",
        r"certified.*self.certified",
    ]
}


# ─────────────────────────────────────────────────────────────
# TEXT PREPROCESSOR CLASS
# ─────────────────────────────────────────────────────────────

class TextPreprocessor:

    def __init__(self):
        try:
            self.stop_words = set(stopwords.words('english')) - {'not', 'no', 'never', 'without', 'free'}
        except:
            self.stop_words = set()
        try:
            self.lemmatizer = WordNetLemmatizer()
        except:
            self.lemmatizer = None
        self.contradiction_patterns = [
            re.compile(p, re.IGNORECASE)
            for p in SUSTAINABILITY_LEXICON["contradiction_phrases"]
        ]

    def clean_text(self, text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return ""
        text = text.lower()
        text = re.sub(r'<[^>]+>', ' ', text)
        text = re.sub(r'http\S+|www\S+', ' ', text)
        contractions = {
            "won't": "will not", "can't": "cannot", "n't": " not",
            "'re": " are", "'ve": " have", "'ll": " will",
            "'d": " would", "'m": " am", "it's": "it is"
        }
        for k, v in contractions.items():
            text = text.replace(k, v)
        text = re.sub(r'(\w)-(\w)', r'\1_\2', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def extract_sustainability_score(self, text):
        text_lower  = text.lower()
        eco_score   = 0.0
        keyword_count      = 0
        has_certification  = False
        has_contradiction  = False
        vague_claim_count  = 0

        for kw, w in SUSTAINABILITY_LEXICON["strong_eco_keywords"].items():
            if kw in text_lower:
                eco_score += w
                keyword_count += 1
                if "certified" in kw or "usda" in kw:
                    has_certification = True

        for kw, w in SUSTAINABILITY_LEXICON["moderate_eco_keywords"].items():
            if kw in text_lower:
                eco_score += w
                keyword_count += 1

        for kw, w in SUSTAINABILITY_LEXICON["greenwashing_indicators"].items():
            if kw in text_lower:
                eco_score += w
                vague_claim_count += 1

        for p in self.contradiction_patterns:
            if p.search(text):
                has_contradiction = True
                eco_score -= 3.0
                break

        return {
            "eco_score"         : round(eco_score, 3),
            "keyword_count"     : keyword_count,
            "has_certification" : int(has_certification),
            "has_contradiction" : int(has_contradiction),
            "vague_claim_count" : vague_claim_count
        }

    def get_text_stats(self, text):
        words = text.split()
        return {
            "word_count"        : len(words),
            "avg_word_length"   : np.mean([len(w) for w in words]) if words else 0,
            "caps_ratio"        : sum(1 for c in text if c.isupper()) / max(len(text), 1),
            "unique_word_ratio" : len(set(words)) / max(len(words), 1)
        }

    def process(self, text):
        cleaned  = self.clean_text(text)
        sus_feat = self.extract_sustainability_score(text)
        txt_stat = self.get_text_stats(cleaned)
        return {"original_text": text, "cleaned_text": cleaned, **sus_feat, **txt_stat}


# ─────────────────────────────────────────────────────────────
# DATASET LOADER CLASS
# ─────────────────────────────────────────────────────────────

class DatasetLoader:

    def __init__(self, base_path="data/raw"):
        self.base_path    = Path(base_path)
        self.preprocessor = TextPreprocessor()

    def load_amazon_reviews(self, sample_size=None):
        path = self.base_path / "amazon_reviews" / "Reviews.csv"
        if not path.exists():
            print(f"  ⚠️  Amazon file not found at {path}")
            return pd.DataFrame()
        df = pd.read_csv(path, nrows=sample_size, low_memory=False)
        df = df.rename(columns={
            'Score': 'rating', 'Text': 'review_text',
            'Summary': 'review_summary', 'Time': 'timestamp',
            'ProductId': 'product_id', 'UserId': 'user_id'
        })
        df['source'] = 'amazon'
        df = df.dropna(subset=['review_text'])
        df['review_text'] = df['review_text'].astype(str)
        df = df[df['review_text'].str.len() >= 10]
        if 'rating' in df.columns:
            df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(3).clip(1, 5).astype(int)
        if 'timestamp' in df.columns:
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce').fillna(pd.Timestamp.now())
        if 'review_summary' not in df.columns:
            df['review_summary'] = df['review_text'].str[:50]
        print(f"  ✅ Amazon loaded: {len(df)} reviews")
        return df

    def load_flipkart_reviews(self, sample_size=None):
        p    = self.base_path / "flipkart_reviews"
        csvs = list(p.glob("*.csv")) if p.exists() else []
        if not csvs:
            print("  ⚠️  Flipkart file not found")
            return pd.DataFrame()
        df = pd.read_csv(csvs[0], nrows=sample_size, low_memory=False)
        df.columns = [c.lower().strip() for c in df.columns]
        for cand in ['review', 'review_text', 'text', 'comment']:
            if cand in df.columns:
                df = df.rename(columns={cand: 'review_text'})
                break
        df['source'] = 'flipkart'
        if 'product_id' not in df.columns:
            for cand in ['product_name', 'product', 'name']:
                if cand in df.columns:
                    df['product_id'] = df[cand]
                    break
            else:
                df['product_id'] = 'unknown'
        if 'user_id' not in df.columns:
            df['user_id'] = [f"fk_{i}" for i in range(len(df))]
        if 'review_summary' not in df.columns:
            df['review_summary'] = df['review_text'].astype(str).str[:50]
        if 'timestamp' not in df.columns:
            df['timestamp'] = pd.Timestamp.now()
        df = df.dropna(subset=['review_text'])
        df['review_text'] = df['review_text'].astype(str)
        df = df[df['review_text'].str.len() >= 10]
        if 'rating' not in df.columns:
            df['rating'] = 3
        df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(3).clip(1, 5).astype(int)
        print(f"  ✅ Flipkart loaded: {len(df)} reviews")
        return df

    def merge_datasets(self, amazon_df, flipkart_df):
        cols = ['product_id', 'user_id', 'source', 'rating',
                'review_text', 'review_summary', 'timestamp', 'is_greenwashing']
        dfs  = []
        for df in [amazon_df, flipkart_df]:
            if df.empty:
                continue
            for c in cols:
                if c not in df.columns:
                    df[c] = None
            dfs.append(df[cols])
        merged = pd.concat(dfs, ignore_index=True)
        merged['review_id']  = range(len(merged))
        merged['timestamp']  = pd.to_datetime(merged['timestamp'], errors='coerce').fillna(pd.Timestamp.now())
        merged['rating']     = pd.to_numeric(merged['rating'], errors='coerce').fillna(3).astype(int)
        merged['review_text'] = merged['review_text'].fillna('').astype(str)
        print(f"  ✅ Merged: {len(merged)} total reviews")
        print(f"     Amazon: {(merged['source']=='amazon').sum()} | Flipkart: {(merged['source']=='flipkart').sum()}")
        return merged

    def preprocess_reviews(self, df):
        print(f"  Processing {len(df)} reviews...")
        features = [self.preprocessor.process(text) for text in df['review_text']]
        fdf      = pd.DataFrame(features)
        df       = df.reset_index(drop=True)
        result   = pd.concat([df, fdf.drop(columns=['original_text'])], axis=1)
        if 'is_greenwashing' not in result.columns or result['is_greenwashing'].isna().all():
            result['is_greenwashing'] = (
                (result['eco_score'] < 1.0) &
                (result['vague_claim_count'] > 0) &
                (result['has_certification'] == 0) &
                (result['keyword_count'] > 0)
            ).astype(int)
        print(f"  ✅ Eco-Friendly: {(result['is_greenwashing']==0).sum()} | Greenwashing: {(result['is_greenwashing']==1).sum()}")
        return result

    def create_timeseries_features(self, df):
        df_sorted  = df.sort_values(['product_id', 'timestamp'])
        ts_records = []
        for pid, pdf in df_sorted.groupby('product_id'):
            if len(pdf) < 2:
                continue
            ratings     = pdf['rating'].values
            eco_scores  = pdf['eco_score'].values if 'eco_score' in pdf.columns else np.zeros(len(pdf))
            rating_std   = np.std(ratings)
            rating_trend = np.polyfit(range(len(ratings)), ratings, 1)[0] if len(ratings) > 1 else 0
            trust_score  = max(0, 1 - (rating_std / 5) - max(0, -rating_trend))
            ts_records.append({
                'product_id'    : pid,
                'review_count'  : len(pdf),
                'mean_rating'   : np.mean(ratings),
                'rating_std'    : rating_std,
                'rating_trend'  : rating_trend,
                'trust_score'   : trust_score,
                'mean_eco_score': np.mean(eco_scores),
                'rating_sequence': ratings.tolist(),
                'eco_sequence'  : eco_scores.tolist()
            })
        print(f"  ✅ Timeseries: {len(ts_records)} products tracked")
        return pd.DataFrame(ts_records)


# ─────────────────────────────────────────────────────────────
# MAIN PIPELINE FUNCTION
# ─────────────────────────────────────────────────────────────

def run_full_preprocessing_pipeline(base_path="data/raw", output_path="data/processed", sample_size=500):
    Path(output_path).mkdir(parents=True, exist_ok=True)
    loader = DatasetLoader(base_path)

    print("  Loading datasets...")
    amazon_df   = loader.load_amazon_reviews(sample_size=sample_size)
    flipkart_df = loader.load_flipkart_reviews(sample_size=sample_size)

    print("  Merging datasets...")
    merged_df = loader.merge_datasets(amazon_df, flipkart_df)

    print("  Preprocessing text...")
    processed_df = loader.preprocess_reviews(merged_df)

    print("  Creating timeseries features...")
    ts_df = loader.create_timeseries_features(processed_df)

    processed_df.to_csv(f"{output_path}/processed_reviews.csv", index=False)
    ts_df.to_csv(f"{output_path}/timeseries_data.csv", index=False)

    print(f"  💾 Saved: {output_path}/processed_reviews.csv")
    print(f"  💾 Saved: {output_path}/timeseries_data.csv")
    return {"reviews": processed_df, "timeseries": ts_df}


# ─────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 4 — Data Preprocessing...")
print("=" * 50)

data         = run_full_preprocessing_pipeline(sample_size=500)
preprocessor = TextPreprocessor()

print()
print("=" * 50)
print("CELL 4 DONE — Data preprocessing complete!")
print("=" * 50)

In [ ]:
# ============================================================
# CELL 5 — NLP MODULE
# TF-IDF + Logistic Regression (CPU fast)
# BERT fine-tuning (GPU recommended)
# Pehle cell4 (preprocessing) zaroor chalao
# ============================================================

import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

try:
    from transformers import BertTokenizer, BertModel
    BERT_AVAILABLE = True
except:
    BERT_AVAILABLE = False

try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except:
    TEXTBLOB_AVAILABLE = False


# ─────────────────────────────────────────────────────────────
# PYTORCH DATASET
# ─────────────────────────────────────────────────────────────

class ReviewDataset(Dataset):

    def __init__(self, texts, labels=None, tokenizer=None, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        if self.tokenizer:
            enc  = self.tokenizer(
                text, max_length=self.max_length,
                padding='max_length', truncation=True, return_tensors='pt'
            )
            item = {
                'input_ids'      : enc['input_ids'].flatten(),
                'attention_mask' : enc['attention_mask'].flatten()
            }
        else:
            item = {'text': text}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# ─────────────────────────────────────────────────────────────
# NLP TRAINER CLASS
# ─────────────────────────────────────────────────────────────

class NLPTrainer:

    def __init__(self):
        self.device          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tfidf_model     = None
        self.tfidf_classifier = None
        print(f"  Device: {self.device} | BERT available: {BERT_AVAILABLE}")

    def get_sentiment_score(self, text):
        if TEXTBLOB_AVAILABLE:
            try:
                return TextBlob(text).sentiment.polarity
            except:
                pass
        pos = sum(1 for w in ['great', 'love', 'excellent', 'amazing', 'certified', 'good'] if w in text.lower())
        neg = sum(1 for w in ['bad', 'terrible', 'fake', 'misleading', 'waste', 'false'] if w in text.lower())
        tot = pos + neg
        return (pos - neg) / tot if tot > 0 else 0.0

    def train_tfidf(self, train_texts, train_labels, val_texts, val_labels):
        print("  Training TF-IDF + Logistic Regression...")

        self.tfidf_model = TfidfVectorizer(
            max_features=8000,
            ngram_range=(1, 3),
            sublinear_tf=True,
            lowercase=True,
            min_df=2,
            max_df=0.95
        )
        X_train = self.tfidf_model.fit_transform(train_texts)
        X_val   = self.tfidf_model.transform(val_texts)

        self.tfidf_classifier = LogisticRegression(
            C=1.0, max_iter=1000,
            class_weight='balanced',
            solver='lbfgs'
        )
        self.tfidf_classifier.fit(X_train, train_labels)

        preds  = self.tfidf_classifier.predict(X_val)
        probs  = self.tfidf_classifier.predict_proba(X_val)[:, 1]

        return {
            'accuracy'               : accuracy_score(val_labels, preds),
            'precision'              : precision_score(val_labels, preds, zero_division=0),
            'recall'                 : recall_score(val_labels, preds, zero_division=0),
            'f1'                     : f1_score(val_labels, preds, zero_division=0),
            'confusion_matrix'       : confusion_matrix(val_labels, preds),
            'classification_report'  : classification_report(
                val_labels, preds,
                target_names=['Genuine Eco', 'Greenwashing'],
                zero_division=0
            ),
            'predictions'  : list(preds),
            'probabilities': list(probs),
            'true_labels'  : list(val_labels)
        }

    def predict(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        results = []
        for text in texts:
            result = {'sentiment_score': self.get_sentiment_score(text)}
            if self.tfidf_model and self.tfidf_classifier:
                feat = self.tfidf_model.transform([text])
                prob = self.tfidf_classifier.predict_proba(feat)[0, 1]
                result.update({
                    'greenwashing_probability': float(prob),
                    'is_greenwashing'         : int(prob > 0.5),
                    'confidence'              : float(max(prob, 1 - prob)),
                    'model_used'              : 'TF-IDF'
                })
            else:
                result.update({
                    'greenwashing_probability': 0.5,
                    'is_greenwashing'         : 0,
                    'confidence'              : 0.5,
                    'model_used'              : 'none'
                })
            results.append(result)
        return results

    def save_model(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump({
                'tfidf'      : self.tfidf_model,
                'classifier' : self.tfidf_classifier
            }, f)
        print(f"  💾 NLP model saved: {path}")

    def load_model(self, path):
        with open(path, 'rb') as f:
            models = pickle.load(f)
        self.tfidf_model      = models['tfidf']
        self.tfidf_classifier = models['classifier']
        print(f"  ✅ NLP model loaded: {path}")


# ─────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────

def train_nlp_module(
    data_path       = "data/processed/processed_reviews.csv",
    model_save_path = "models/nlp/nlp_model.pkl"
):
    df     = pd.read_csv(data_path)
    texts  = df['cleaned_text'].fillna('').tolist()
    labels = df['is_greenwashing'].fillna(0).astype(int).tolist()

    X_temp, X_test, y_temp, y_test = train_test_split(
        texts, labels, test_size=0.15, random_state=42, stratify=labels
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
    )
    print(f"  Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    trainer = NLPTrainer()
    metrics = trainer.train_tfidf(X_train, y_train, X_val, y_val)
    trainer.save_model(model_save_path)

    print(f"  Accuracy:  {metrics['accuracy']:.3f}")
    print(f"  Precision: {metrics['precision']:.3f}")
    print(f"  Recall:    {metrics['recall']:.3f}")
    print(f"  F1 Score:  {metrics['f1']:.3f}")
    print()
    print(metrics['classification_report'])

    return {'trainer': trainer, 'metrics': metrics}


# ─────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 5 — NLP Module Training...")
print("=" * 50)

nlp_results = train_nlp_module()

print()
print("=" * 50)
print("CELL 5 DONE — NLP module trained!")
print("=" * 50)


In [ ]:
# ============================================================
# CELL 6 — CNN MODULE
# ResNet50 Transfer Learning — Packaging Classification
# Eco-Friendly vs Non-Eco Packaging
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import ResNet50_Weights

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ─────────────────────────────────────────────────────────────
# IMAGE DATASET
# ─────────────────────────────────────────────────────────────

class PackagingDataset(Dataset):

    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path  = self.paths[idx]
        label = self.labels[idx]
        try:
            if isinstance(path, str) and Path(path).exists():
                img = Image.open(path).convert('RGB')
            else:
                img = self._synthetic(label)
        except:
            img = self._synthetic(label)
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

    def _synthetic(self, label):
        arr = np.zeros((224, 224, 3), dtype=np.uint8)
        c   = (34, 139, 34) if label == 0 else (128, 128, 128)
        for i in range(3):
            arr[:, :, i] = np.clip(c[i] + np.random.randint(-30, 30, (224, 224)), 0, 255)
        return Image.fromarray(arr)


# ─────────────────────────────────────────────────────────────
# CNN MODEL — ResNet50 + Custom Head
# ─────────────────────────────────────────────────────────────

class PackagingCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Load pretrained ResNet50
        self.backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # Freeze early layers — only train layer4 + custom head
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False

        # Replace final FC layer
        self.backbone.fc = nn.Identity()

        # Custom classification head
        self.head = nn.Sequential(
            nn.BatchNorm1d(2048),
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)


# ─────────────────────────────────────────────────────────────
# CNN TRAINER CLASS
# ─────────────────────────────────────────────────────────────

class CNNTrainer:

    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = None
        print(f"  Device: {self.device}")

    def setup_model(self):
        self.model = PackagingCNN().to(self.device)
        tr = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"  Trainable parameters: {tr:,}")

    def load_image_data(self, data_path="data/raw/waste_images"):
        paths  = []
        labels = []
        data_dir = Path(data_path)
        cls_map  = {'O': 0, 'R': 0, 'N': 1, 'Organic': 0, 'Recyclable': 0, 'Trash': 1}

        if data_dir.exists():
            for split in ['DATASET/TRAIN', 'DATASET/TEST', '']:
                sd = data_dir / split
                if not sd.exists():
                    continue
                for cd in sd.iterdir():
                    if not cd.is_dir():
                        continue
                    label = cls_map.get(cd.name, None)
                    if label is None:
                        for k, v in cls_map.items():
                            if k.lower() in cd.name.lower():
                                label = v
                                break
                    if label is None:
                        continue
                    for ext in ['*.jpg', '*.jpeg', '*.png']:
                        for f in cd.glob(ext):
                            paths.append(str(f))
                            labels.append(label)

        if not paths:
            print("  ⚠️  No images found — using synthetic images")
            paths  = [f"syn_{i}" for i in range(120)]
            labels = [i % 2 for i in range(120)]

        print(f"  Images: {len(paths)} | Eco: {sum(1 for l in labels if l==0)} | Non-eco: {sum(1 for l in labels if l==1)}")
        return paths, labels

    def train(self, data_path="data/raw/waste_images", epochs=5):
        self.setup_model()
        paths, labels = self.load_image_data(data_path)

        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]

        train_tf = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])
        val_tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std)
        ])

        X_tr, X_te, y_tr, y_te = train_test_split(paths, labels, test_size=0.2, random_state=42, stratify=labels)
        X_tr, X_v,  y_tr, y_v  = train_test_split(X_tr, y_tr,   test_size=0.2, random_state=42, stratify=y_tr)

        tr_ld  = DataLoader(PackagingDataset(X_tr, y_tr, train_tf), batch_size=16, shuffle=True)
        val_ld = DataLoader(PackagingDataset(X_v,  y_v,  val_tf),   batch_size=32)
        te_ld  = DataLoader(PackagingDataset(X_te, y_te, val_tf),   batch_size=32)

        optimizer  = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=1e-4, weight_decay=1e-4
        )
        scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion  = nn.CrossEntropyLoss()
        best_acc   = 0
        best_state = None

        for ep in range(epochs):
            self.model.train()
            tr_correct = 0
            tr_total   = 0
            for imgs, lbls in tr_ld:
                imgs, lbls = imgs.to(self.device), lbls.to(self.device)
                optimizer.zero_grad()
                out  = self.model(imgs)
                loss = criterion(out, lbls)
                loss.backward()
                optimizer.step()
                tr_correct += out.argmax(1).eq(lbls).sum().item()
                tr_total   += lbls.size(0)
            val_m = self._eval(val_ld)
            scheduler.step()
            print(f"  Epoch {ep+1}/{epochs} | Train Acc: {tr_correct/tr_total:.3f} | Val Acc: {val_m['accuracy']:.3f}")
            if val_m['accuracy'] > best_acc:
                best_acc   = val_m['accuracy']
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}

        if best_state:
            self.model.load_state_dict(best_state)

        return self._eval(te_ld)

    def _eval(self, loader):
        self.model.eval()
        preds  = []
        labels = []
        probs  = []
        with torch.no_grad():
            for imgs, lbls in loader:
                imgs = imgs.to(self.device)
                out  = self.model(imgs)
                pr   = torch.softmax(out, 1)
                preds  += out.argmax(1).cpu().tolist()
                labels += lbls.tolist()
                probs  += pr[:, 1].cpu().tolist()
        return {
            'accuracy'              : accuracy_score(labels, preds),
            'confusion_matrix'      : confusion_matrix(labels, preds),
            'classification_report' : classification_report(
                labels, preds,
                target_names=['Eco-Friendly', 'Non-Eco'],
                zero_division=0
            ),
            'predictions'  : preds,
            'probabilities': probs,
            'true_labels'  : labels
        }

    def predict_image(self, image):
        if self.model is None:
            return {'error': 'Model not loaded'}
        if isinstance(image, str):
            img = Image.open(image).convert('RGB')
        elif isinstance(image, np.ndarray):
            img = Image.fromarray(image.astype(np.uint8))
        else:
            img = image.convert('RGB')
        tf = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        t = tf(img).unsqueeze(0).to(self.device)
        self.model.eval()
        with torch.no_grad():
            out   = self.model(t)
            probs = torch.softmax(out, 1)
            cls   = out.argmax(1).item()
            conf  = probs.max(1)[0].item()
        return {
            'class'               : ['eco_friendly', 'non_eco'][cls],
            'class_id'            : cls,
            'confidence'          : round(conf, 4),
            'eco_probability'     : round(probs[0, 0].item(), 4),
            'non_eco_probability' : round(probs[0, 1].item(), 4),
            'is_eco_friendly'     : cls == 0
        }

    def save_model(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({'model_state_dict': self.model.state_dict()}, path)
        print(f"  💾 CNN model saved: {path}")

    def load_model(self, path):
        self.setup_model()
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        print(f"  ✅ CNN model loaded: {path}")


# ─────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────

def train_cnn_module(
    data_path       = "data/raw/waste_images",
    model_save_path = "models/cnn/resnet_packaging_model.pt",
    epochs          = 5
):
    trainer  = CNNTrainer()
    test_m   = trainer.train(data_path, epochs=epochs)
    trainer.save_model(model_save_path)

    print(f"  Test Accuracy: {test_m['accuracy']:.3f}")
    print()
    print(test_m['classification_report'])

    return {'trainer': trainer, 'metrics': test_m}


# ─────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 6 — CNN Module Training...")
print("=" * 50)

cnn_results = train_cnn_module(epochs=5)

print()
print("=" * 50)
print("CELL 6 DONE — CNN module trained!")
print("=" * 50)


In [ ]:
# ============================================================
# CELL 7 — RNN MODULE
# Bidirectional LSTM + Temporal Attention
# Review sequence pattern detection — Trust Score
# Pehle cell4 (preprocessing) zaroor chalao
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score


# ─────────────────────────────────────────────────────────────
# SEQUENCE DATASET
# ─────────────────────────────────────────────────────────────

class SeqDataset(Dataset):

    def __init__(self, seqs, labels, max_len=50):
        self.seqs    = seqs
        self.labels  = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq   = self.seqs[idx]
        label = self.labels[idx]
        if len(seq.shape) == 1:
            seq = seq.reshape(-1, 1)
        T       = min(seq.shape[0], self.max_len)
        seq     = seq[-T:]
        padded  = np.zeros((self.max_len, seq.shape[1]), dtype=np.float32)
        padded[:T] = seq
        mask    = np.zeros(self.max_len, dtype=np.float32)
        mask[:T]   = 1
        return {
            'sequence': torch.FloatTensor(padded),
            'mask'    : torch.FloatTensor(mask),
            'length'  : T,
            'label'   : torch.FloatTensor([float(label)])
        }


# ─────────────────────────────────────────────────────────────
# TEMPORAL ATTENTION
# Kaunsi review sabse suspicious hai?
# ─────────────────────────────────────────────────────────────

class TemporalAttention(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, lstm_out, mask):
        scores  = self.attn(lstm_out).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=1)
        context = (weights.unsqueeze(-1) * lstm_out).sum(dim=1)
        return context, weights


# ─────────────────────────────────────────────────────────────
# BILSTM MODEL
# ─────────────────────────────────────────────────────────────

class LSTMTrustScorer(nn.Module):

    def __init__(self, input_size=5, hidden=64, layers=2):
        super().__init__()

        # Input projection
        self.proj = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.LayerNorm(32),
            nn.ReLU()
        )

        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size  = 32,
            hidden_size = hidden,
            num_layers  = layers,
            batch_first = True,
            bidirectional = True,
            dropout     = 0.3 if layers > 1 else 0
        )

        # Attention
        self.attn = TemporalAttention(hidden * 2)

        # Output head — Trust Score [0, 1]
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x, mask, lengths=None):
        xp              = self.proj(x)
        lstm_out, _     = self.lstm(xp)
        context, weights = self.attn(lstm_out, mask)
        trust_score     = self.head(context).squeeze(-1)
        return {
            'trust_score'      : trust_score,
            'attention_weights': weights
        }


# ─────────────────────────────────────────────────────────────
# RNN TRAINER CLASS
# ─────────────────────────────────────────────────────────────

class RNNTrainer:

    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = None
        print(f"  Device: {self.device}")

    def generate_synthetic_sequences(self, n=300):
        np.random.seed(42)
        seqs   = []
        labels = []
        for i in range(n):
            pattern = np.random.choice(['trust', 'greenwash', 'astro'], p=[0.4, 0.3, 0.3])
            L       = np.random.randint(5, 30)
            t       = np.linspace(0, 1, L)
            if pattern == 'trust':
                r   = np.clip(0.7 + 0.1 * np.random.randn(L), 0, 1)
                lbl = 0.8 + 0.1 * np.random.random()
            elif pattern == 'greenwash':
                r   = np.clip(0.9 - 0.4 * t + 0.1 * np.random.randn(L), 0, 1)
                lbl = 0.2 + 0.15 * np.random.random()
            else:
                r   = np.where(t < 0.2, 0.95, np.clip(0.4 + 0.2 * np.random.randn(L), 0, 1))
                lbl = 0.15 + 0.15 * np.random.random()
            e   = np.clip(r * 0.8 + 0.1 * np.random.randn(L), 0, 1)
            s   = np.clip(r * 0.9 + 0.1 * np.random.randn(L), 0, 1)
            k   = np.clip(e * 2, 0, 1)
            c   = np.random.choice([0, 1], L, p=[0.7, 0.3])
            seq = np.column_stack([r, e, s, k, c]).astype(np.float32)
            seqs.append(seq)
            labels.append(float(np.clip(lbl, 0, 1)))
        return seqs, labels

    def train(self, seqs, labels, epochs=30):
        self.model = LSTMTrustScorer().to(self.device)

        X_tr, X_te, y_tr, y_te = train_test_split(seqs, labels, test_size=0.2, random_state=42)
        X_tr, X_v,  y_tr, y_v  = train_test_split(X_tr, y_tr,   test_size=0.2, random_state=42)

        tr_ld  = DataLoader(SeqDataset(X_tr, y_tr), batch_size=32, shuffle=True)
        val_ld = DataLoader(SeqDataset(X_v,  y_v),  batch_size=64)
        te_ld  = DataLoader(SeqDataset(X_te, y_te), batch_size=64)

        optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        criterion = nn.MSELoss()

        best      = float('inf')
        best_state = None
        patience  = 0

        for ep in range(epochs):
            self.model.train()
            losses = []
            for b in tr_ld:
                seq  = b['sequence'].to(self.device)
                mask = b['mask'].to(self.device)
                lbl  = b['label'].to(self.device).squeeze()
                optimizer.zero_grad()
                out  = self.model(seq, mask)
                loss = criterion(out['trust_score'], lbl)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            val_m = self._eval(val_ld)
            scheduler.step(val_m['mse'])
            if ep % 10 == 0:
                print(f"  Epoch {ep:3d}/{epochs} | Loss: {np.mean(losses):.4f} | Val MSE: {val_m['mse']:.4f} | R²: {val_m['r2']:.3f}")
            if val_m['mse'] < best:
                best       = val_m['mse']
                best_state = {k: v.clone() for k, v in self.model.state_dict().items()}
                patience   = 0
            else:
                patience += 1
                if patience >= 10:
                    print(f"  Early stopping at epoch {ep}")
                    break

        if best_state:
            self.model.load_state_dict(best_state)

        return self._eval(te_ld)

    def _eval(self, loader):
        self.model.eval()
        preds  = []
        labels = []
        with torch.no_grad():
            for b in loader:
                seq  = b['sequence'].to(self.device)
                mask = b['mask'].to(self.device)
                lbl  = b['label'].to(self.device).squeeze()
                out  = self.model(seq, mask)
                preds  += out['trust_score'].cpu().tolist()
                labels += lbl.cpu().tolist()
        p  = np.array(preds)
        l  = np.array(labels)
        pb = (p > 0.5).astype(int)
        lb = (l > 0.5).astype(int)
        return {
            'mse'        : float(mean_squared_error(l, p)),
            'r2'         : float(r2_score(l, p)) if len(set(l)) > 1 else 0,
            'accuracy'   : float(accuracy_score(lb, pb)),
            'predictions': p,
            'true_labels': l
        }

    def predict_trust_score(self, seq_list):
        results = []
        for sd in seq_list:
            ratings = sd.get('ratings', [3, 3, 3])
            eco     = sd.get('eco_scores', [0] * len(ratings))
            n       = len(ratings)
            feat    = np.column_stack([
                np.array(ratings) / 5.0,
                np.clip((np.array(eco) + 5) / 10, 0, 1),
                np.zeros(n), np.zeros(n), np.zeros(n)
            ]).astype(np.float32)
            ds  = SeqDataset([feat], [0.0])
            ld  = DataLoader(ds, batch_size=1)
            self.model.eval()
            with torch.no_grad():
                b   = next(iter(ld))
                out = self.model(b['sequence'].to(self.device), b['mask'].to(self.device))
                ts  = out['trust_score'].item()
                aw  = out['attention_weights'][0].cpu().numpy()
            if ts >= 0.7:   interp, risk = "Highly Trustworthy", "Low"
            elif ts >= 0.5: interp, risk = "Moderately Trustworthy", "Medium"
            elif ts >= 0.3: interp, risk = "Suspicious Pattern", "High"
            else:           interp, risk = "Likely Greenwashing", "Critical"
            results.append({
                'trust_score'      : round(ts, 4),
                'interpretation'   : interp,
                'risk_level'       : risk,
                'attention_weights': aw[:n].tolist()
            })
        return results

    def save_model(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({'model_state_dict': self.model.state_dict()}, path)
        print(f"  💾 RNN model saved: {path}")

    def load_model(self, path):
        self.model = LSTMTrustScorer().to(self.device)
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        print(f"  ✅ RNN model loaded: {path}")


# ─────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────

def train_rnn_module(
    ts_data_path    = "data/processed/timeseries_data.csv",
    model_save_path = "models/rnn/lstm_trust_model.pt",
    epochs          = 30
):
    trainer       = RNNTrainer()
    seqs, labels  = trainer.generate_synthetic_sequences(n=500)
    test_m        = trainer.train(seqs, labels, epochs=epochs)
    trainer.save_model(model_save_path)

    print(f"  MSE:      {test_m['mse']:.4f}")
    print(f"  R²:       {test_m['r2']:.4f}")
    print(f"  Accuracy: {test_m['accuracy']:.3f}")

    return {'trainer': trainer, 'metrics': test_m}


# ─────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 7 — RNN Module Training...")
print("=" * 50)

rnn_results = train_rnn_module(epochs=30)

print()
print("=" * 50)
print("CELL 7 DONE — RNN module trained!")
print("=" * 50)

In [ ]:
# ============================================================
# CELL 8 — FINAL ENSEMBLE MODEL
# XGBoost Meta-Learner
# NLP + CNN + RNN outputs combine karke final score nikalta hai
# Pehle cells 5, 6, 7 zaroor chalao
# ============================================================

import pickle
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False
    print("  ⚠️  XGBoost not found — using GradientBoosting")


# ─────────────────────────────────────────────────────────────
# 20 FEATURE NAMES
# ─────────────────────────────────────────────────────────────

FEATURE_NAMES = [
    'nlp_greenwash_prob',
    'nlp_sentiment',
    'nlp_eco_score',
    'nlp_keyword_count',
    'nlp_has_certification',
    'nlp_has_contradiction',
    'nlp_vague_claims',
    'nlp_confidence',
    'cnn_eco_prob',
    'cnn_non_eco_prob',
    'cnn_confidence',
    'cnn_is_eco',
    'rnn_trust_score',
    'rnn_risk_encoded',
    'rnn_suspicious_count',
    'rnn_attn_variance',
    'inter_nlp_cnn_agree',
    'inter_nlp_trust',
    'inter_eco_signal',
    'inter_contradiction'
]


# ─────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# Sab modules ka output ek vector mein combine
# ─────────────────────────────────────────────────────────────

def engineer_features(nlp, cnn=None, rnn=None):
    risk_enc = {'Low': 0.0, 'Medium': 0.33, 'High': 0.67, 'Critical': 1.0}
    nlp_prob = float(nlp.get('greenwashing_probability', 0.5))

    # NLP features (8)
    feats = [
        nlp_prob,
        float(nlp.get('sentiment_score', 0)),
        float(nlp.get('eco_score', 0)),
        float(nlp.get('keyword_count', 0)) / 10,
        float(nlp.get('has_certification', 0)),
        float(nlp.get('has_contradiction', 0)),
        float(nlp.get('vague_claim_count', 0)) / 5,
        float(nlp.get('confidence', 0.5))
    ]

    # CNN features (4)
    if cnn and 'error' not in cnn:
        ep  = float(cnn.get('eco_probability', 0.5))
        nep = float(cnn.get('non_eco_probability', 0.5))
        feats += [ep, nep, float(cnn.get('confidence', 0.5)), float(1 - cnn.get('class_id', 0))]
    else:
        feats += [0.5, 0.5, 0.5, 0.5]

    # RNN features (4)
    if rnn and 'error' not in rnn:
        ts   = float(rnn.get('trust_score', 0.5))
        risk = rnn.get('risk_level', 'Medium')
        susp = rnn.get('suspicious_timestamps', [])
        aw   = rnn.get('attention_weights', [0.5])
        feats += [
            ts,
            risk_enc.get(risk, 0.33),
            min(float(len(susp)) / 10, 1.0),
            float(np.var(aw)) if aw else 0
        ]
    else:
        feats += [0.5, 0.33, 0.0, 0.0]

    # Interaction features (4)
    cnn_sig = float(cnn.get('non_eco_probability', 0.5)) if cnn and 'error' not in cnn else 0.5
    trust   = feats[12]
    agree   = 1 - abs(nlp_prob - cnn_sig)
    nlp_tr  = nlp_prob * (1 - trust)
    eco_sig = feats[1] * 0.2 + (1 - nlp_prob) * 0.3 + feats[8] * 0.2 + trust * 0.3
    contra  = feats[5] * 0.5 + feats[6] * 0.3 + (1 - agree) * 0.2
    feats  += [agree, nlp_tr, eco_sig, contra]

    return np.array(feats, dtype=np.float32)


# ─────────────────────────────────────────────────────────────
# ENSEMBLE CLASS
# ─────────────────────────────────────────────────────────────

class GreenwashEnsemble:

    def __init__(self):
        self.meta_model   = None
        self.scaler       = StandardScaler()
        self.feature_names = FEATURE_NAMES
        self.is_trained   = False

    def train(self, X, y):
        Xs = self.scaler.fit_transform(X)
        X_tr, X_te, y_tr, y_te = train_test_split(
            Xs, y, test_size=0.2, random_state=42, stratify=y
        )

        if XGB_AVAILABLE:
            sp = sum(y_tr == 0) / max(sum(y_tr == 1), 1)
            self.meta_model = xgb.XGBClassifier(
                n_estimators      = 200,
                max_depth         = 6,
                learning_rate     = 0.05,
                subsample         = 0.8,
                colsample_bytree  = 0.8,
                scale_pos_weight  = sp,
                eval_metric       = 'logloss',
                random_state      = 42,
                verbosity         = 0
            )
            self.meta_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
        else:
            self.meta_model = GradientBoostingClassifier(
                n_estimators = 200, max_depth=4,
                learning_rate=0.05, random_state=42
            )
            self.meta_model.fit(X_tr, y_tr)

        preds = self.meta_model.predict(X_te)
        probs = self.meta_model.predict_proba(X_te)[:, 1]

        cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_f1   = cross_val_score(self.meta_model, Xs, y, cv=cv, scoring='f1')

        metrics = {
            'accuracy'              : accuracy_score(y_te, preds),
            'precision'             : precision_score(y_te, preds, zero_division=0),
            'recall'                : recall_score(y_te, preds, zero_division=0),
            'f1'                    : f1_score(y_te, preds, zero_division=0),
            'auc'                   : roc_auc_score(y_te, probs) if len(set(y_te)) > 1 else 0,
            'cv_f1_mean'            : cv_f1.mean(),
            'cv_f1_std'             : cv_f1.std(),
            'confusion_matrix'      : confusion_matrix(y_te, preds),
            'classification_report' : classification_report(
                y_te, preds,
                target_names=['Eco-Friendly', 'Greenwashing'],
                zero_division=0
            ),
            'true_labels'  : list(y_te),
            'probabilities': list(probs)
        }

        self.is_trained = True
        return metrics

    def predict(self, nlp_output, cnn_output=None, rnn_output=None):
        feats = engineer_features(nlp_output, cnn_output, rnn_output)
        fs    = self.scaler.transform(feats.reshape(1, -1))

        if self.meta_model and self.is_trained:
            probs = self.meta_model.predict_proba(fs)[0]
            gs    = float(probs[1])
            conf  = float(max(probs))
        else:
            gs   = float(nlp_output.get('greenwashing_probability', 0.5))
            conf = 0.5

        if gs < 0.2:   risk, verdict = "Very Low",  "Genuinely Eco-Friendly"
        elif gs < 0.4: risk, verdict = "Low",        "Likely Eco-Friendly"
        elif gs < 0.6: risk, verdict = "Moderate",   "Uncertain — Needs More Analysis"
        elif gs < 0.8: risk, verdict = "High",       "Likely Greenwashing"
        else:          risk, verdict = "Critical",   "Strong Greenwashing Indicators"

        cls = "Greenwashing" if gs > 0.5 else "Eco-Friendly"

        return {
            'greenwashing_score'      : round(gs, 4),
            'greenwashing_percentage' : round(gs * 100, 1),
            'classification'          : cls,
            'verdict'                 : verdict,
            'risk_level'              : risk,
            'confidence'              : round(conf, 4),
            'eco_score'               : round(1 - gs, 4),
            'module_breakdown': {
                'nlp_contribution': round(float(nlp_output.get('greenwashing_probability', 0.5)), 3),
                'cnn_contribution': round(float(cnn_output.get('non_eco_probability', 0.5)) if cnn_output else 0.5, 3),
                'rnn_contribution': round(1 - float(rnn_output.get('trust_score', 0.5)) if rnn_output else 0.5, 3)
            }
        }

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump({
                'meta_model'   : self.meta_model,
                'scaler'       : self.scaler,
                'feature_names': self.feature_names,
                'is_trained'   : self.is_trained
            }, f)
        print(f"  💾 Ensemble model saved: {path}")

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        self.meta_model    = data['meta_model']
        self.scaler        = data['scaler']
        self.feature_names = data['feature_names']
        self.is_trained    = data['is_trained']
        print(f"  ✅ Ensemble model loaded: {path}")


# ─────────────────────────────────────────────────────────────
# MAIN TRAINING FUNCTION
# ─────────────────────────────────────────────────────────────

def train_final_model(
    data_path  = "data/processed/processed_reviews.csv",
    model_path = "models/final/greenwash_ensemble.pkl"
):
    df = pd.read_csv(data_path)

    if 'greenwash_prob' not in df.columns:
        df['greenwash_prob'] = df['is_greenwashing'].map({0: 0.2, 1: 0.8}) + \
                               np.random.uniform(-0.1, 0.1, len(df))
        df['greenwash_prob'] = df['greenwash_prob'].clip(0, 1)
    if 'sentiment_score' not in df.columns:
        df['sentiment_score'] = np.random.uniform(-0.5, 0.5, len(df))
    if 'trust_score' not in df.columns:
        df['trust_score'] = (1 - df['is_greenwashing']) + np.random.uniform(-0.2, 0.2, len(df))
        df['trust_score'] = df['trust_score'].clip(0, 1)

    ens = GreenwashEnsemble()
    X   = []
    y   = []

    for _, row in df.iterrows():
        nlp = {
            'greenwashing_probability': float(row.get('greenwash_prob', 0.5)),
            'sentiment_score'         : float(row.get('sentiment_score', 0)),
            'eco_score'               : float(row.get('eco_score', 0)),
            'keyword_count'           : int(row.get('keyword_count', 0)),
            'has_certification'       : int(row.get('has_certification', 0)),
            'has_contradiction'       : int(row.get('has_contradiction', 0)),
            'vague_claim_count'       : int(row.get('vague_claim_count', 0)),
            'confidence'              : 0.7
        }
        cnn = {
            'eco_probability'     : np.random.uniform(0.3, 0.7),
            'non_eco_probability' : np.random.uniform(0.3, 0.7),
            'confidence'          : np.random.uniform(0.5, 0.9),
            'class_id'            : np.random.randint(0, 2)
        }
        rnn = {
            'trust_score'         : float(row.get('trust_score', 0.5)),
            'risk_level'          : 'Medium',
            'suspicious_timestamps': [],
            'attention_weights'   : [0.5]
        }
        X.append(engineer_features(nlp, cnn, rnn))
        y.append(int(row.get('is_greenwashing', 0)))

    X = np.array(X)
    y = np.array(y)

    print(f"  Features: {X.shape} | Labels: {np.bincount(y)}")
    metrics = ens.train(X, y)
    ens.save(model_path)

    print(f"  Accuracy:  {metrics['accuracy']:.3f}")
    print(f"  Precision: {metrics['precision']:.3f}")
    print(f"  Recall:    {metrics['recall']:.3f}")
    print(f"  F1 Score:  {metrics['f1']:.3f}")
    print(f"  AUC-ROC:   {metrics['auc']:.3f}")
    print(f"  CV F1:     {metrics['cv_f1_mean']:.3f} ± {metrics['cv_f1_std']:.3f}")
    print()
    print(metrics['classification_report'])

    return {'ensemble': ens, 'metrics': metrics}


# ─────────────────────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 8 — Final Ensemble Model Training...")
print("=" * 50)

ensemble_results = train_final_model()

print()
print("=" * 50)
print("CELL 8 DONE — Ensemble model trained!")
print("=" * 50)

In [ ]:
# ============================================================
# CELL 9 — EDA CHARTS
# Distribution, Word Clouds, Sustainability Features
# CELL 10 — EVALUATION REPORT
# Confusion Matrix, ROC Curve, All Module Scores
# Pehle cell4 aur cell8 zaroor chalao
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
Path("reports").mkdir(exist_ok=True)

# ─────────────────────────────────────────────────────────────
# CELL 9 — EDA CHARTS
# ─────────────────────────────────────────────────────────────

print("=" * 50)
print("CELL 9 — EDA Charts...")
print("=" * 50)

df = pd.read_csv("data/processed/processed_reviews.csv")
print(f"  Loaded: {len(df)} reviews")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Greenwashing Detector — EDA Dashboard', fontsize=16, fontweight='bold')

# 1. Class Balance
ax = axes[0, 0]
counts = df['is_greenwashing'].value_counts()
bars = ax.bar(
    ['Eco-Friendly', 'Greenwashing'],
    [counts.get(0, 0), counts.get(1, 0)],
    color=['#2ecc71', '#e74c3c'], width=0.5
)
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
for bar, cnt in zip(bars, [counts.get(0, 0), counts.get(1, 0)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
            f'{cnt}\n({cnt/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=10)

# 2. Rating Distribution
ax = axes[0, 1]
rc = df['rating'].value_counts().sort_index()
ax.bar(rc.index, rc.values,
       color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'])
ax.set_title('Rating Distribution', fontweight='bold')
ax.set_xlabel('Stars')
ax.set_ylabel('Count')

# 3. Eco Score by Class
ax = axes[0, 2]
ax.hist(df[df['is_greenwashing'] == 0]['eco_score'], bins=20, alpha=0.7, color='#2ecc71', label='Eco-Friendly')
ax.hist(df[df['is_greenwashing'] == 1]['eco_score'], bins=20, alpha=0.7, color='#e74c3c', label='Greenwashing')
ax.axvline(0, color='black', linestyle='--', alpha=0.5)
ax.legend()
ax.set_title('Eco Score Distribution', fontweight='bold')
ax.set_xlabel('Eco Score')

# 4. Keyword Count by Class
ax = axes[1, 0]
df.boxplot(column='keyword_count', by='is_greenwashing', ax=ax,
           boxprops=dict(color='#2c3e50'), medianprops=dict(color='#e74c3c'))
ax.set_title('Keyword Count by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Count')
ax.set_xticklabels(['Eco-Friendly', 'Greenwashing'])
plt.sca(ax)
plt.title('Keyword Count by Class')

# 5. Certification vs Class
ax = axes[1, 1]
ct = pd.crosstab(df['has_certification'], df['is_greenwashing'])
ct.columns = ['Eco-Friendly', 'Greenwashing']
ct.index   = ['No Cert', 'Has Cert']
ct.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'],
        edgecolor='white', width=0.7, rot=0)
ax.set_title('Certification vs Class', fontweight='bold')
ax.legend()

# 6. Source Split
ax = axes[1, 2]
if 'source' in df.columns:
    sc = df['source'].value_counts()
    ax.pie(sc.values, labels=sc.index,
           colors=['#3498db', '#9b59b6'],
           autopct='%1.1f%%', startangle=90)
    ax.set_title('Data Source Split', fontweight='bold')

plt.tight_layout()
plt.savefig("reports/eda_dashboard.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
print("  ✅ Saved: reports/eda_dashboard.png")

# Word Clouds
try:
    from wordcloud import WordCloud, STOPWORDS
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Word Clouds: Eco-Friendly vs Greenwashing Reviews', fontsize=14, fontweight='bold')
    custom_stop = set(STOPWORDS) | {'product', 'item', 'buy', 'bought', 'use', 'used', 'one', 'get'}
    for idx, (lbl, title, cmap) in enumerate([(0, 'Eco-Friendly Reviews', 'Greens'),
                                               (1, 'Greenwashing Reviews', 'Reds')]):
        text = ' '.join(df[df['is_greenwashing'] == lbl]['cleaned_text'].fillna(''))
        if text.strip():
            wc = WordCloud(width=700, height=380, background_color='white',
                           colormap=cmap, stopwords=custom_stop, max_words=80).generate(text)
            axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].set_title(title, fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    plt.tight_layout()
    plt.savefig("reports/word_clouds.png", dpi=130, bbox_inches='tight', facecolor='white')
    plt.show()
    print("  ✅ Saved: reports/word_clouds.png")
except ImportError:
    print("  ⚠️  wordcloud not installed — skipping word clouds")

print()
print("=" * 50)
print("CELL 9 DONE — EDA charts ready!")
print("=" * 50)


# ─────────────────────────────────────────────────────────────
# CELL 10 — EVALUATION REPORT
# ─────────────────────────────────────────────────────────────

print()
print("=" * 50)
print("CELL 10 — Evaluation Report...")
print("=" * 50)

from sklearn.metrics import roc_curve, auc, precision_recall_curve

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Evaluation Report', fontsize=16, fontweight='bold')

# 1. NLP Confusion Matrix
cm = nlp_results['metrics']['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn_r',
            xticklabels=['Eco', 'Greenwash'],
            yticklabels=['Eco', 'Greenwash'],
            ax=axes[0, 0])
axes[0, 0].set_title('NLP Confusion Matrix', fontweight='bold')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')

# 2. All Module Scores
ax = axes[0, 1]
mets = {
    'NLP\nF1'       : nlp_results['metrics']['f1'],
    'CNN\nAccuracy' : cnn_results['metrics']['accuracy'],
    'RNN\nR²'       : max(0, rnn_results['metrics']['r2']),
    'Ensemble\nF1'  : ensemble_results['metrics']['f1']
}
bars = ax.bar(mets.keys(), mets.values(),
              color=['#3498db', '#9b59b6', '#e67e22', '#2ecc71'], width=0.5)
ax.set_ylim(0, 1.15)
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Target 0.80')
ax.legend()
ax.set_title('All Module Performance', fontweight='bold')
ax.set_ylabel('Score')
for bar, val in zip(bars, mets.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold')

# 3. ROC Curves
ax = axes[1, 0]
for res, name, color in [
    (nlp_results,      'NLP',      '#3498db'),
    (ensemble_results, 'Ensemble', '#2ecc71')
]:
    tl = res['metrics'].get('true_labels', [])
    pr = res['metrics'].get('probabilities', [])
    if len(tl) > 0 and len(set(tl)) > 1:
        fpr, tpr, _ = roc_curve(tl, pr)
        ra = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={ra:.3f})')
        ax.fill_between(fpr, tpr, alpha=0.08, color=color)
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
ax.set_title('ROC Curves', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()

# 4. Ensemble Confusion Matrix
cm2 = ensemble_results['metrics']['confusion_matrix']
sns.heatmap(cm2, annot=True, fmt='d', cmap='RdYlGn_r',
            xticklabels=['Eco', 'Greenwash'],
            yticklabels=['Eco', 'Greenwash'],
            ax=axes[1, 1])
axes[1, 1].set_title('Ensemble Confusion Matrix', fontweight='bold')
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig("reports/evaluation_report.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.show()
print("  ✅ Saved: reports/evaluation_report.png")

# Final Summary Table
print()
print("=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  NLP (TF-IDF+LR)   Acc: {nlp_results['metrics']['accuracy']:.3f}  |  F1: {nlp_results['metrics']['f1']:.3f}")
print(f"  CNN (ResNet50)     Acc: {cnn_results['metrics']['accuracy']:.3f}")
print(f"  RNN (BiLSTM)       MSE: {rnn_results['metrics']['mse']:.4f} |  R²: {rnn_results['metrics']['r2']:.4f}")
print(f"  Ensemble (Final)   Acc: {ensemble_results['metrics']['accuracy']:.3f}  |  F1: {ensemble_results['metrics']['f1']:.3f}  |  AUC: {ensemble_results['metrics']['auc']:.3f}")
print("=" * 60)

print()
print("=" * 50)
print("CELL 10 DONE — Evaluation complete!")
print("=" * 50)

In [ ]:
# ============================================================
# CELL 11 — LIVE PREDICTION TEST
# Apna khud ka review paste karo aur result dekho
# Pehle tamam cells (1-10) zaroor chalao
# ============================================================

print("=" * 60)
print("CELL 11 — LIVE GREENWASHING DETECTION TEST")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# 3 Test Reviews — Alag alag types
# ─────────────────────────────────────────────────────────────

test_reviews = [
    {
        "expected" : "✅ GENUINE ECO",
        "review"   : "This shampoo is USDA certified organic. B-Corp certified company. Packaging is 100% compostable. Rainforest Alliance verified ingredients. Genuinely sustainable!"
    },
    {
        "expected" : "🚨 GREENWASHING",
        "review"   : "All natural and chemical free! Our pure earth-friendly shampoo is eco and bio with green packaging. Going green together with our sustainable choice!"
    },
    {
        "expected" : "⚠️  AMBIGUOUS",
        "review"   : "Claims to be sustainable and uses organic ingredients. Packaging says eco-friendly but I could not find any certifications anywhere on the product."
    }
]

print()

for item in test_reviews:
    review = item["review"]

    # Step 1: Preprocess
    processed = preprocessor.process(review)

    # Step 2: NLP prediction
    nlp_out = nlp_results['trainer'].predict([review])[0]
    nlp_out.update({
        'eco_score'         : processed['eco_score'],
        'keyword_count'     : processed['keyword_count'],
        'has_certification' : processed['has_certification'],
        'has_contradiction' : processed['has_contradiction'],
        'vague_claim_count' : processed['vague_claim_count'],
    })

    # Step 3: Final ensemble prediction
    result = ensemble_results['ensemble'].predict(nlp_output=nlp_out)

    # Display
    score = result['greenwashing_score']
    if score < 0.35:   icon = "✅"
    elif score < 0.65: icon = "⚠️ "
    else:               icon = "🚨"

    print(f"Expected  : {item['expected']}")
    print(f"Review    : {review[:80]}...")
    print(f"{icon} Score    : {result['greenwashing_percentage']:.1f}%  |  Risk: {result['risk_level']}")
    print(f"   Verdict : {result['verdict']}")
    print(f"   NLP     : {nlp_out['greenwashing_probability']*100:.1f}%  |  Sentiment: {nlp_out['sentiment_score']:+.2f}")
    print(f"   Keywords: {processed['keyword_count']} eco  |  Vague claims: {processed['vague_claim_count']}")
    print("-" * 60)

print()

# ─────────────────────────────────────────────────────────────
# CUSTOM TEST — Apna review yahan paste karo
# ─────────────────────────────────────────────────────────────

print("CUSTOM TEST — Apna review yahan paste karo:")
print()

your_review = "This product uses eco-friendly materials and natural organic ingredients. Sustainable packaging."

processed_custom = preprocessor.process(your_review)
nlp_custom       = nlp_results['trainer'].predict([your_review])[0]
nlp_custom.update({
    'eco_score'         : processed_custom['eco_score'],
    'keyword_count'     : processed_custom['keyword_count'],
    'has_certification' : processed_custom['has_certification'],
    'has_contradiction' : processed_custom['has_contradiction'],
    'vague_claim_count' : processed_custom['vague_claim_count'],
})
result_custom = ensemble_results['ensemble'].predict(nlp_output=nlp_custom)

print(f"Review  : {your_review}")
print(f"Score   : {result_custom['greenwashing_percentage']:.1f}%")
print(f"Verdict : {result_custom['verdict']}")
print(f"Risk    : {result_custom['risk_level']}")

print()
print("=" * 50)
print("CELL 11 DONE — Live prediction complete!")
print("=" * 50)


# ============================================================
# CELL 12 — SAVE TO GOOGLE DRIVE (Optional)
# Models aur reports Google Drive mein save karo
# Taake Colab restart hone pe na jayein
# ============================================================

print()
print("=" * 50)
print("CELL 12 — Save to Google Drive...")
print("=" * 50)

from google.colab import drive
import shutil

drive.mount('/content/drive')

shutil.copytree("models",  "/content/drive/MyDrive/greenwash_models",  dirs_exist_ok=True)
shutil.copytree("reports", "/content/drive/MyDrive/greenwash_reports", dirs_exist_ok=True)
shutil.copytree("data/processed", "/content/drive/MyDrive/greenwash_data", dirs_exist_ok=True)

print("  ✅ models/   → /MyDrive/greenwash_models/")
print("  ✅ reports/  → /MyDrive/greenwash_reports/")
print("  ✅ data/processed/ → /MyDrive/greenwash_data/")
print()
print("=" * 50)
print("CELL 12 DONE — Everything saved to Google Drive!")
print("=" * 50)

In [ ]:
# ============================================================
# STREAMLIT APP — Greenwashing Detector
# Run karne ka tarika Colab mein:
#
# !pip install streamlit pyngrok -q
# from pyngrok import ngrok
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")  # ngrok.com se free token lo
# !streamlit run streamlit_app.py &
# public_url = ngrok.connect(8501)
# print(public_url)
# ============================================================

import streamlit as st
import numpy as np
import pandas as pd
import sys
import pickle
from pathlib import Path
from PIL import Image
import plotly.graph_objects as go
import plotly.express as px

# ─────────────────────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────────────────────

st.set_page_config(
    page_title="🌿 GreenCheck — Greenwashing Detector",
    page_icon="🌿",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ─────────────────────────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────────────────────────

st.markdown("""
<style>
    .header-box {
        background: linear-gradient(135deg, #1a472a, #2d6a4f);
        padding: 2rem; border-radius: 12px;
        margin-bottom: 1.5rem;
    }
    .header-box h1 { color: white !important; font-size: 2.2rem !important; margin: 0 !important; }
    .header-box p  { color: #d8f3dc !important; margin: 0.3rem 0 0 0 !important; }

    .score-card {
        background: white; border-radius: 10px;
        padding: 1.2rem; text-align: center;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    }
    .verdict-eco      { background: #d4edda; color: #155724; padding: 0.8rem 1.2rem; border-radius: 8px; font-weight: 600; text-align: center; }
    .verdict-greenwash{ background: #f8d7da; color: #721c24; padding: 0.8rem 1.2rem; border-radius: 8px; font-weight: 600; text-align: center; }
    .verdict-warning  { background: #fff3cd; color: #856404; padding: 0.8rem 1.2rem; border-radius: 8px; font-weight: 600; text-align: center; }

    .eco-badge  { display: inline-block; background: #d4edda; color: #155724; padding: 3px 10px; border-radius: 12px; margin: 2px; font-size: 0.82rem; font-weight: 600; }
    .gw-badge   { display: inline-block; background: #f8d7da; color: #721c24; padding: 3px 10px; border-radius: 12px; margin: 2px; font-size: 0.82rem; font-weight: 600; }

    #MainMenu {visibility: hidden;}
    footer {visibility: hidden;}
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────
# LOAD MODELS (cached)
# ─────────────────────────────────────────────────────────────

@st.cache_resource(show_spinner=False)
def load_all_models():
    """Models load karo — sirf ek baar"""

    # Import all classes
    # Note: Colab mein cells ke baad yeh classes available hoti hain
    # Agar standalone run karo to import karo

    try:
        preprocessor_obj = TextPreprocessor()
    except NameError:
        # Fallback minimal preprocessor
        class MinimalPreprocessor:
            def process(self, text):
                import re
                cleaned = re.sub(r'[^\w\s]', ' ', text.lower()).strip()
                eco_kws = ['certified', 'organic', 'biodegradable', 'recycled', 'compostable', 'solar', 'renewable']
                gw_kws  = ['all natural', 'chemical free', 'pure', 'earth friendly']
                eco_score   = sum(2 for k in eco_kws if k in text.lower())
                gw_score    = sum(1 for k in gw_kws if k in text.lower())
                return {
                    'cleaned_text'     : cleaned,
                    'eco_score'        : eco_score - gw_score,
                    'keyword_count'    : eco_score,
                    'has_certification': int('certified' in text.lower()),
                    'has_contradiction': 0,
                    'vague_claim_count': gw_score
                }
        preprocessor_obj = MinimalPreprocessor()

    # Load NLP model
    nlp_trainer_obj = None
    nlp_path = Path("models/nlp/nlp_model.pkl")
    if nlp_path.exists():
        try:
            nlp_trainer_obj = NLPTrainer()
            nlp_trainer_obj.load_model(str(nlp_path))
        except:
            pass

    # Load CNN model
    cnn_trainer_obj = None
    cnn_path = Path("models/cnn/resnet_packaging_model.pt")
    if cnn_path.exists():
        try:
            cnn_trainer_obj = CNNTrainer()
            cnn_trainer_obj.load_model(str(cnn_path))
        except:
            pass

    # Load Ensemble model
    ens_obj  = None
    ens_path = Path("models/final/greenwash_ensemble.pkl")
    if ens_path.exists():
        try:
            ens_obj = GreenwashEnsemble()
            ens_obj.load(str(ens_path))
        except:
            pass

    return preprocessor_obj, nlp_trainer_obj, cnn_trainer_obj, ens_obj


# ─────────────────────────────────────────────────────────────
# HELPER: RUN FULL PREDICTION
# ─────────────────────────────────────────────────────────────

def run_prediction(text, preprocessor_obj, nlp_trainer_obj, ens_obj, image=None, cnn_trainer_obj=None):
    """Full prediction pipeline"""

    processed = preprocessor_obj.process(text)

    # NLP prediction
    if nlp_trainer_obj:
        nlp_out = nlp_trainer_obj.predict([text])[0]
    else:
        # Simple fallback
        eco_s = processed['eco_score']
        prob  = max(0, min(1, 0.5 - eco_s * 0.1))
        nlp_out = {
            'greenwashing_probability': prob,
            'sentiment_score'         : 0.0,
            'confidence'              : 0.6,
            'is_greenwashing'         : int(prob > 0.5)
        }

    nlp_out.update({
        'eco_score'         : processed['eco_score'],
        'keyword_count'     : processed['keyword_count'],
        'has_certification' : processed['has_certification'],
        'has_contradiction' : processed['has_contradiction'],
        'vague_claim_count' : processed['vague_claim_count'],
    })

    # CNN prediction
    cnn_out = None
    if image and cnn_trainer_obj:
        cnn_out = cnn_trainer_obj.predict_image(image)

    # Final ensemble prediction
    if ens_obj:
        result = ens_obj.predict(nlp_output=nlp_out, cnn_output=cnn_out)
    else:
        # Fallback scoring
        score   = float(nlp_out['greenwashing_probability'])
        if score < 0.2:   risk, verdict = "Very Low",  "Genuinely Eco-Friendly"
        elif score < 0.4: risk, verdict = "Low",       "Likely Eco-Friendly"
        elif score < 0.6: risk, verdict = "Moderate",  "Uncertain"
        elif score < 0.8: risk, verdict = "High",      "Likely Greenwashing"
        else:             risk, verdict = "Critical",  "Strong Greenwashing Indicators"
        result = {
            'greenwashing_score'      : round(score, 4),
            'greenwashing_percentage' : round(score * 100, 1),
            'classification'          : "Greenwashing" if score > 0.5 else "Eco-Friendly",
            'verdict'                 : verdict,
            'risk_level'              : risk,
            'confidence'              : 0.6,
            'eco_score'               : round(1 - score, 4),
            'module_breakdown'        : {
                'nlp_contribution': round(score, 3),
                'cnn_contribution': 0.5,
                'rnn_contribution': 0.5
            }
        }

    return result, nlp_out, processed, cnn_out


# ─────────────────────────────────────────────────────────────
# GAUGE CHART
# ─────────────────────────────────────────────────────────────

def make_gauge(score, title="Greenwashing Risk"):
    color = "#2ecc71" if score < 0.35 else ("#f39c12" if score < 0.65 else "#e74c3c")
    fig   = go.Figure(go.Indicator(
        mode  = "gauge+number",
        value = score * 100,
        title = {'text': title, 'font': {'size': 13}},
        number= {'suffix': "%", 'font': {'size': 26, 'color': color}},
        gauge = {
            'axis'     : {'range': [0, 100]},
            'bar'      : {'color': color, 'thickness': 0.28},
            'steps'    : [
                {'range': [0, 30],  'color': '#d4edda'},
                {'range': [30, 60], 'color': '#fff3cd'},
                {'range': [60, 100],'color': '#f8d7da'}
            ],
            'threshold': {'line': {'color': '#2c3e50', 'width': 3}, 'value': 50}
        }
    ))
    fig.update_layout(
        height=200,
        margin=dict(l=10, r=10, t=30, b=10),
        paper_bgcolor='rgba(0,0,0,0)'
    )
    return fig


# ─────────────────────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────────────────────

with st.sidebar:
    st.markdown("""
    <div style="text-align:center; padding:1rem 0">
        <div style="font-size:2.5rem">🌿</div>
        <h2 style="color:#2d6a4f; margin:0">GreenCheck</h2>
        <p style="color:#666; font-size:0.85rem">AI Greenwashing Detector</p>
    </div>
    """, unsafe_allow_html=True)
    st.divider()

    page = st.radio(
        "Navigate",
        ["🔍 Analyze Review", "🖼️ Image Analysis", "📊 Dashboard", "ℹ️ How It Works"],
        label_visibility="collapsed"
    )

    st.divider()
    st.markdown("**Model Status**")
    for name, path in [
        ("NLP Model",      "models/nlp/nlp_model.pkl"),
        ("CNN Model",      "models/cnn/resnet_packaging_model.pt"),
        ("RNN Model",      "models/rnn/lstm_trust_model.pt"),
        ("Ensemble Model", "models/final/greenwash_ensemble.pkl")
    ]:
        exists = Path(path).exists()
        st.markdown(f"{'✅' if exists else '⚠️'} {name}")


# ─────────────────────────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────────────────────────

with st.spinner("Loading AI models..."):
    try:
        preprocessor_m, nlp_trainer_m, cnn_trainer_m, ensemble_m = load_all_models()
        models_ok = True
    except Exception as e:
        st.warning(f"Model loading issue: {e} — running in demo mode")
        preprocessor_m = None
        nlp_trainer_m  = None
        cnn_trainer_m  = None
        ensemble_m     = None
        models_ok      = False


# ─────────────────────────────────────────────────────────────
# PAGE 1 — ANALYZE REVIEW
# ─────────────────────────────────────────────────────────────

if page == "🔍 Analyze Review":

    st.markdown("""
    <div class="header-box">
        <h1>🔍 Greenwashing Detector</h1>
        <p>Paste any product review — AI will detect greenwashing in seconds</p>
    </div>
    """, unsafe_allow_html=True)

    samples = {
        "Select a sample..."             : "",
        "✅ Genuine Eco (Good example)"  : "This bamboo toothbrush is USDA certified organic. B-Corp certified company, truly sustainable. Packaging is 100% compostable and plastic-free.",
        "🚨 Greenwashing (Bad example)"  : "All natural and chemical free! Our pure earth-friendly product is eco and bio with green packaging. Going green with our sustainable choice!",
        "⚠️  Ambiguous (Mixed signals)"  : "Claims to be sustainable and uses organic ingredients. Packaging says eco-friendly but no certifications visible anywhere on product."
    }

    col1, col2 = st.columns([2, 1])

    with col1:
        sample  = st.selectbox("Try a sample:", list(samples.keys()))
        review  = st.text_area(
            "Product review:",
            value   = samples[sample],
            height  = 140,
            placeholder="Paste product review here..."
        )
        analyze = st.button("🔍 Analyze for Greenwashing", type="primary", use_container_width=True)

    with col2:
        st.markdown("**Optional: Upload Product Image**")
        uploaded = st.file_uploader(
            "Upload packaging image:",
            type=['jpg', 'jpeg', 'png'],
            label_visibility="collapsed"
        )
        if uploaded:
            img = Image.open(uploaded)
            st.image(img, use_column_width=True, caption="Uploaded Image")
        else:
            st.info("📷 No image uploaded\nText-only analysis will run")

    if analyze and review and len(review) > 10:

        if preprocessor_m is None:
            st.error("Models not loaded. Please run training cells first.")
        else:
            with st.spinner("🤖 Running AI analysis..."):
                image_obj = Image.open(uploaded) if uploaded else None
                result, nlp_out, processed, cnn_out = run_prediction(
                    review, preprocessor_m, nlp_trainer_m,
                    ensemble_m, image_obj, cnn_trainer_m
                )

            st.divider()
            score = result['greenwashing_score']

            # Verdict Banner
            if score < 0.35:
                st.markdown(f'<div class="verdict-eco">✅ {result["verdict"].upper()} — {result["risk_level"]} RISK</div>', unsafe_allow_html=True)
            elif score < 0.65:
                st.markdown(f'<div class="verdict-warning">⚠️ {result["verdict"].upper()} — {result["risk_level"]} RISK</div>', unsafe_allow_html=True)
            else:
                st.markdown(f'<div class="verdict-greenwash">🚨 {result["verdict"].upper()} — {result["risk_level"]} RISK</div>', unsafe_allow_html=True)

            st.markdown("")

            # Scores row
            c1, c2, c3, c4 = st.columns(4)
            with c1:
                st.plotly_chart(make_gauge(score), use_container_width=True)
            with c2:
                eco = result['eco_score']
                color = "#2ecc71" if eco > 0.6 else ("#f39c12" if eco > 0.4 else "#e74c3c")
                st.markdown(f"""
                <div class="score-card" style="margin-top:0.5rem">
                    <div style="font-size:0.85rem;color:#888">Eco Score</div>
                    <div style="font-size:2.2rem;color:{color};font-weight:800">{eco*100:.0f}%</div>
                    <div style="font-size:0.75rem;color:#aaa">Authenticity</div>
                </div>""", unsafe_allow_html=True)
            with c3:
                sent = nlp_out.get('sentiment_score', 0)
                icon = "😊" if sent > 0.2 else ("😐" if sent > -0.2 else "😞")
                st.markdown(f"""
                <div class="score-card" style="margin-top:0.5rem">
                    <div style="font-size:0.85rem;color:#888">Sentiment</div>
                    <div style="font-size:2rem">{icon}</div>
                    <div style="font-size:1rem;font-weight:700">{sent:+.2f}</div>
                </div>""", unsafe_allow_html=True)
            with c4:
                kw = processed['keyword_count']
                kw_color = "#2ecc71" if kw > 2 else ("#f39c12" if kw > 0 else "#e74c3c")
                st.markdown(f"""
                <div class="score-card" style="margin-top:0.5rem">
                    <div style="font-size:0.85rem;color:#888">Eco Keywords</div>
                    <div style="font-size:2.2rem;color:{kw_color};font-weight:800">{kw}</div>
                    <div style="font-size:0.75rem;color:#aaa">Found</div>
                </div>""", unsafe_allow_html=True)

            st.markdown("")

            # Module Breakdown
            bd = result['module_breakdown']
            fig = go.Figure(go.Bar(
                x  = [v * 100 for v in bd.values()],
                y  = ['🧠 NLP (BERT)', '🖼️ CNN (ResNet)', '🔄 RNN (LSTM)'],
                orientation = 'h',
                marker_color = ['#3498db', '#9b59b6', '#e67e22'],
                text = [f"{v*100:.1f}%" for v in bd.values()],
                textposition = 'outside'
            ))
            fig.update_layout(
                title="Module Contribution to Risk Score",
                xaxis=dict(range=[0, 115], title="Greenwashing Risk (%)"),
                height=200,
                margin=dict(l=10, r=60, t=40, b=10),
                paper_bgcolor='rgba(0,0,0,0)'
            )
            st.plotly_chart(fig, use_container_width=True)

            # Keywords
            with st.expander("🔑 Keyword Analysis", expanded=True):
                eco_kws = ['certified organic', 'usda organic', 'b corp', 'biodegradable',
                           'compostable', 'recycled', 'renewable', 'fair trade', 'plastic free']
                gw_kws  = ['all natural', 'chemical free', 'pure', 'earth friendly', 'eco', 'bio', 'going green']

                found_eco = [k for k in eco_kws if k in review.lower()]
                found_gw  = [k for k in gw_kws  if k in review.lower()]

                col_a, col_b, col_c = st.columns(3)
                with col_a:
                    st.markdown("**✅ Eco Keywords:**")
                    if found_eco:
                        st.markdown(' '.join([f'<span class="eco-badge">{k}</span>' for k in found_eco]), unsafe_allow_html=True)
                    else:
                        st.caption("None found")
                with col_b:
                    st.markdown("**⚠️ Vague Claims:**")
                    if found_gw:
                        st.markdown(' '.join([f'<span class="gw-badge">{k}</span>' for k in found_gw]), unsafe_allow_html=True)
                    else:
                        st.caption("None found")
                with col_c:
                    st.markdown("**📋 Flags:**")
                    if processed['has_certification']:
                        st.markdown("✅ Has eco certification")
                    if processed['has_contradiction']:
                        st.markdown("🚨 Contradictory claims!")
                    if processed['vague_claim_count'] > 2:
                        st.markdown("⚠️ Many vague claims")
                    if not any([processed['has_certification'], processed['has_contradiction'], processed['vague_claim_count'] > 2]):
                        st.caption("No specific flags")

    elif analyze:
        st.warning("Please enter at least 10 characters.")


# ─────────────────────────────────────────────────────────────
# PAGE 2 — IMAGE ANALYSIS
# ─────────────────────────────────────────────────────────────

elif page == "🖼️ Image Analysis":

    st.markdown("""
    <div class="header-box">
        <h1>🖼️ Packaging Image Analyzer</h1>
        <p>Upload a product image — CNN will classify packaging as eco-friendly or non-eco</p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2 = st.columns([1, 1])

    with col1:
        uploaded_img = st.file_uploader("Upload packaging image", type=['jpg', 'jpeg', 'png'])
        if uploaded_img:
            img = Image.open(uploaded_img)
            st.image(img, caption="Uploaded Image", use_column_width=True)
            if st.button("🖼️ Classify Packaging", type="primary", use_container_width=True):
                if cnn_trainer_m:
                    with st.spinner("Running CNN..."):
                        res = cnn_trainer_m.predict_image(img)
                    st.session_state['cnn_result'] = res
                else:
                    st.error("CNN model not loaded. Run cell6_cnn_module.py first.")

    with col2:
        if 'cnn_result' in st.session_state:
            res = st.session_state['cnn_result']
            if res['is_eco_friendly']:
                st.success("✅ ECO-FRIENDLY PACKAGING")
            else:
                st.error("🚨 NON-ECO PACKAGING")

            fig = go.Figure(go.Bar(
                x=['🌿 Eco-Friendly', '🗑️ Non-Eco'],
                y=[res['eco_probability']*100, res['non_eco_probability']*100],
                marker_color=['#2ecc71', '#e74c3c'],
                text=[f"{res['eco_probability']*100:.1f}%", f"{res['non_eco_probability']*100:.1f}%"],
                textposition='outside'
            ))
            fig.update_layout(
                yaxis=dict(range=[0, 120], title="Probability (%)"),
                height=280, showlegend=False,
                margin=dict(l=10, r=10, t=10, b=10),
                paper_bgcolor='rgba(0,0,0,0)'
            )
            st.plotly_chart(fig, use_container_width=True)
            st.metric("Confidence", f"{res['confidence']*100:.1f}%")
        else:
            st.info("Upload an image and click classify to see results.")


# ─────────────────────────────────────────────────────────────
# PAGE 3 — DASHBOARD
# ─────────────────────────────────────────────────────────────

elif page == "📊 Dashboard":

    st.markdown("""
    <div class="header-box">
        <h1>📊 ESG Analytics Dashboard</h1>
        <p>Product sustainability performance and greenwashing trends</p>
    </div>
    """, unsafe_allow_html=True)

    # Generate sample dashboard data
    np.random.seed(42)
    n   = 200
    ddf = pd.DataFrame({
        'product'            : [f'Product {i}' for i in range(n)],
        'category'           : np.random.choice(['Skincare', 'Food', 'Cleaning', 'Clothing', 'Electronics'], n),
        'greenwashing_score' : np.random.beta(2, 3, n),
        'eco_score'          : np.random.beta(3, 2, n),
        'trust_score'        : np.random.beta(4, 2, n),
        'has_certification'  : np.random.choice([0, 1], n, p=[0.7, 0.3]),
        'review_count'       : np.random.randint(10, 500, n),
    })
    ddf['classification'] = ddf['greenwashing_score'].apply(
        lambda x: 'Greenwashing' if x > 0.5 else 'Eco-Friendly'
    )

    # KPIs
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.metric("Products Analyzed", n)
    c2.metric("Greenwashing Rate", f"{(ddf['greenwashing_score']>0.5).mean()*100:.1f}%")
    c3.metric("Avg Eco Score",     f"{ddf['eco_score'].mean()*100:.1f}%")
    c4.metric("Certified Products",f"{ddf['has_certification'].mean()*100:.1f}%")
    c5.metric("Avg Trust Score",   f"{ddf['trust_score'].mean()*100:.1f}%")

    st.divider()

    col1, col2 = st.columns(2)

    with col1:
        cat = ddf.groupby('category')['greenwashing_score'].mean().sort_values()
        fig = px.bar(x=cat.values*100, y=cat.index, orientation='h',
                     title="Greenwashing Rate by Category",
                     labels={'x': 'Score (%)', 'y': 'Category'},
                     color=cat.values, color_continuous_scale='RdYlGn_r')
        fig.update_layout(height=300, coloraxis_showscale=False,
                          paper_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        cc = ddf['classification'].value_counts()
        fig = go.Figure(go.Pie(
            labels=cc.index, values=cc.values, hole=0.5,
            marker_colors=['#2ecc71', '#e74c3c']
        ))
        fig.update_layout(title="Product Split",
                          height=300, paper_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig, use_container_width=True)

    # Scatter
    fig = px.scatter(ddf, x='eco_score', y='trust_score',
                     color='greenwashing_score',
                     color_continuous_scale='RdYlGn_r',
                     size='review_count', hover_data=['category'],
                     title="Eco Score vs Trust Score")
    fig.update_layout(height=350, paper_bgcolor='rgba(0,0,0,0)')
    st.plotly_chart(fig, use_container_width=True)


# ─────────────────────────────────────────────────────────────
# PAGE 4 — HOW IT WORKS
# ─────────────────────────────────────────────────────────────

elif page == "ℹ️ How It Works":

    st.markdown("""
    <div class="header-box">
        <h1>ℹ️ How GreenCheck Works</h1>
        <p>Three AI models working together to detect greenwashing</p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("""
        ### 🧠 NLP Module
        **Model:** TF-IDF + Logistic Regression (BERT optional)
        - Cleans and tokenizes text
        - Detects 75+ eco keywords
        - Identifies certifications
        - Flags contradictory claims
        - Sentiment analysis
        - **Output:** Greenwashing probability 0–1
        """)

    with col2:
        st.markdown("""
        ### 🖼️ CNN Module
        **Model:** ResNet50 Transfer Learning
        - ImageNet pretrained weights
        - layer4 fine-tuned
        - Image augmentation
        - Custom classification head
        - **Output:** Eco vs Non-Eco packaging probability
        """)

    with col3:
        st.markdown("""
        ### 🔄 RNN Module
        **Model:** Bidirectional LSTM + Attention
        - Review sequences over time
        - Detects rating drift patterns
        - Astroturfing detection
        - Temporal attention weights
        - **Output:** Trust Score 0–1
        """)

    st.divider()

    st.markdown("""
    ### 📊 Greenwashing Score Interpretation

    | Score | Risk | Verdict |
    |-------|------|---------|
    | 0 – 20% | 🟢 Very Low | Genuinely Eco-Friendly |
    | 20 – 40% | 🟡 Low | Likely Eco-Friendly |
    | 40 – 60% | 🟠 Moderate | Needs More Analysis |
    | 60 – 80% | 🔴 High | Likely Greenwashing |
    | 80 – 100% | ⛔ Critical | Strong Greenwashing Indicators |
    """)


# ─────────────────────────────────────────────────────────────
# RUN INSTRUCTIONS (shown at bottom)
# ─────────────────────────────────────────────────────────────

st.sidebar.divider()
st.sidebar.caption("""
**Run in Colab:**
```python
!pip install streamlit pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_TOKEN")
!streamlit run streamlit_app.py &
url = ngrok.connect(8501)
print(url)
```
""")